# 미들웨어와 6개 훅

미들웨어의 여섯 훅은 에이전트 실행의 어느 지점에서 어떤 순서로 호출될까요?


## 실행 환경 준비

실습 파일은 `labs/part6/ch22-middleware/notebooks/01-미들웨어와-6개-훅.ipynb`입니다. Python 3.12 새 커널에서 위에서부터 실행합니다.


## 패키지 버전 확인


In [1]:
# 이 셀은 패키지 버전 확인 작업을 수행하고 확인할 값을 출력합니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langgraph==1.2.11" "python-dotenv==1.2.3"
from importlib.metadata import version
print("langchain", version("langchain"))
print("langchain-openai", version("langchain-openai"))
print("langgraph", version("langgraph"))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
langgraph 1.2.11


## 실제 API 연결과 비용 확인


In [2]:
# 이 셀은 실제 API 연결과 비용 확인 작업을 수행하고 확인할 값을 출력합니다.
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file(): load_dotenv(folder / ".env"); break
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."
live = init_chat_model("openai:gpt-5.4-mini", reasoning_effort="none")
reply = live.invoke("LangChain 에이전트 미들웨어를 한국어 한 문장으로 정의하세요.")
usage = reply.usage_metadata or {}
cost = usage.get("input_tokens",0)/1_000_000*0.75 + usage.get("output_tokens",0)/1_000_000*4.50
print("answer:", reply.text)
print("usage:", usage)
print("estimated_cost_usd:", round(cost,6))
assert cost < 1.0

answer: LangChain 에이전트 미들웨어는 에이전트의 동작 과정에서 요청·응답·도구 호출 등을 가로채어 검증, 변환, 로깅, 제어를 수행하는 중간 처리 계층입니다.
usage: {'input_tokens': 25, 'output_tokens': 57, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
estimated_cost_usd: 0.000275


## 훅 확인용 결정적 모델 준비


In [3]:
# 이 셀은 훅 확인용 결정적 모델 준비 작업을 수행하고 확인할 값을 출력합니다.
# 정해 둔 응답으로 미들웨어 실행 순서를 재현합니다.
from typing import Any
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
class ScriptedModel(BaseChatModel):
    replies: list[AIMessage]
    index: int = 0
    @property
    def _llm_type(self) -> str: return "scripted-model"
    def _generate(self, messages: Any, stop: Any = None, run_manager: Any = None, **kwargs: Any) -> ChatResult:
        reply = self.replies[min(self.index, len(self.replies) - 1)]
        self.index += 1
        return ChatResult(generations=[ChatGeneration(message=reply)])
    def bind_tools(self, tools: Any, *, tool_choice: Any = None, **kwargs: Any) -> "ScriptedModel":
        return self
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

## 도구와 여섯 훅 trace 준비

미들웨어는 `create_agent`가 만든 루프의 약속된 시점에 실행되는 로직입니다. 개발자가 훅 함수를 직접 호출하지 않아도 런타임이 `before_agent`, `before_model`, `after_model`, `after_agent`를 노드처럼 실행합니다. `wrap_model_call`과 `wrap_tool_call`은 대상 호출과 `handler`를 함께 받아 호출 전후를 감쌉니다. 랩 훅에서 `handler`를 호출하지 않으면 실제 모델 또는 도구도 실행되지 않으므로 단순한 전후 알림과 제어 로직을 구분해야 합니다.

`SixHookTrace`는 상태를 바꾸지 않고 `TRACE`에 이름만 남기는 관찰용 클래스입니다. 첫 모델 응답이 도구 호출을 만들기 때문에 `before_model`과 모델 랩은 두 번 실행되고, 도구 랩은 한 번 실행됩니다. 반면 `before_agent`와 `after_agent`는 한 번의 `invoke` 경계에서 각각 한 번 나타납니다. 훅의 횟수는 등록 개수가 아니라 실제 루프 반복과 도구 선택에 따라 달라집니다.


In [4]:
# 이 셀은 도구와 여섯 훅 trace 준비 작업을 수행하고 확인할 값을 출력합니다.
from langchain.agents import create_agent
from langchain.agents.middleware import AgentMiddleware
from langchain.tools import tool

TRACE = []
@tool
def lookup_hours(day: str) -> str:
    """자료실 운영 시간을 요일로 조회합니다."""
    return "평일 09:00~20:00" if day == "평일" else "토요일 10:00~17:00"

class SixHookTrace(AgentMiddleware):
    def before_agent(self, state: Any, runtime: Any) -> None: TRACE.append("before_agent")
    def before_model(self, state: Any, runtime: Any) -> None: TRACE.append("before_model")
    def wrap_model_call(self, request: Any, handler: Any) -> Any:
        TRACE.append("wrap_model:before"); result = handler(request); TRACE.append("wrap_model:after"); return result
    def wrap_tool_call(self, request: Any, handler: Any) -> Any:
        TRACE.append("wrap_tool:before"); result = handler(request); TRACE.append("wrap_tool:after"); return result
    def after_model(self, state: Any, runtime: Any) -> None: TRACE.append("after_model")
    def after_agent(self, state: Any, runtime: Any) -> None: TRACE.append("after_agent")

## 도구 1회인 에이전트 실행


In [5]:
# 이 셀은 도구 1회인 에이전트 실행 작업을 수행하고 확인할 값을 출력합니다.
scripted = ScriptedModel(replies=[
    AIMessage(content="", tool_calls=[{"name":"lookup_hours","args":{"day":"평일"},"id":"call_1","type":"tool_call"}]),
    AIMessage(content="평일 운영 시간은 09:00~20:00입니다."),
])
agent = create_agent(scripted, tools=[lookup_hours], middleware=[SixHookTrace()])
result = agent.invoke({"messages":[{"role":"user","content":"평일 운영 시간은?"}]})
print("trace:", TRACE)
print("answer:", result["messages"][-1].text)
assert TRACE.count("before_model") == 2 and TRACE.count("wrap_tool:before") == 1

trace: ['before_agent', 'before_model', 'wrap_model:before', 'wrap_model:after', 'after_model', 'wrap_tool:before', 'wrap_tool:after', 'before_model', 'wrap_model:before', 'wrap_model:after', 'after_model', 'after_agent']
answer: 평일 운영 시간은 09:00~20:00입니다.


## 훅 호출 횟수 집계

모델 입력에 사용자 등급을 추가하는 일은 `before_model`, 일시 오류를 재시도하는 일은 `wrap_model_call`, 도구 인자를 검사하는 일은 `wrap_tool_call`이 자연스럽습니다. 같은 검사를 모든 노드에 복사하면 변경 위치가 늘어나므로 반복되는 횡단 관심사를 미들웨어로 모읍니다.

모든 로직을 미들웨어로 옮길 필요는 없습니다. 주문 금액 계산처럼 업무 자체를 나타내는 단계는 그래프 노드나 도구에 두는 편이 흐름을 읽기 쉽습니다. 이 실습의 TRACE 순서는 한 번의 도구 호출 경로에서 관찰한 값이며, 도구가 없거나 병렬 호출이 있으면 횟수와 순서가 달라질 수 있습니다.


In [6]:
# 이 셀은 훅 호출 횟수 집계 작업을 수행하고 확인할 값을 출력합니다.
from collections import Counter
counts = Counter(item.split(":")[0] for item in TRACE)
print(dict(counts))
assert TRACE[0] == "before_agent" and TRACE[-1] == "after_agent"

{'before_agent': 1, 'before_model': 2, 'wrap_model': 4, 'after_model': 2, 'wrap_tool': 2, 'after_agent': 1}
